<a href="https://colab.research.google.com/github/ocayaro/PLA-PCL-polymer-chain/blob/main/pla-pcl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import os

# Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

# Install OpenMM, OpenFF, and dependencies via mamba
!mamba install -c conda-forge openmm openmmforcefields openff-toolkit rdkit mdanalysis matplotlib -y
!mamba install -c conda-forge openff-packmol -y

# Ensure Conda site-packages is in Python's search path *before* further imports
# This often needs to happen after condacolab.install() to ensure the correct path is found.
conda_prefix = os.environ.get('CONDA_PREFIX', '/usr/local')
conda_site_packages = os.path.join(conda_prefix, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
if conda_site_packages not in sys.path:
    sys.path.append(conda_site_packages)
    print(f"Added {conda_site_packages} to sys.path to ensure conda packages are found.")

# Fallback to pip install if mamba install doesn't resolve the ModuleNotFoundError.
# Note: A runtime restart and re-running all cells will still be required after this.
!pip install -q openff-packmol

# Explicitly install rdkit as a fallback if mamba or previous pip fails
!pip install -q rdkit

# Verifying openff-packmol installation
print('Verifying openff-packmol installation:')
!pip list | grep openff-packmol

print('Verifying rdkit installation:')
!pip list | grep rdkit

# Robust check for RDKit import
try:
    import rdkit
    print('\n[SUCCESS] RDKit is successfully installed and importable within this cell.')
except ImportError:
    print('\n[ERROR] RDKit could not be imported after installation attempts.')
    print('This often happens if the Python environment is not fully refreshed.')

print("""\n--- IMPORTANT: PLEASE RESTART THE RUNTIME (Runtime > Restart runtime) AND THEN 'Run all' CELLS! --- Some packages may conflict with pre-installed Colab packages.
Only after a full restart and re-running all cells will the new installations be correctly recognized by all parts of the notebook.\n""")

# Fix for torchvision::nms RuntimeError: Uninstall and reinstall compatible torch/torchvision
print('\n[FIX] Attempting to resolve torchvision conflict by reinstalling torch/torchvision...')
!pip uninstall -y torch torchvision torchaudio
# Install a common compatible version of torch and torchvision for Colab (assuming CUDA 12.1)
# You might need to adjust 'cu121' based on your Colab GPU runtime version
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print('[FIX] Reinstallation of torch/torchvision completed.')

# Fix for openff.units ImportError: Cannot import name 'get_current_backend'
print('\n[FIX] Attempting to resolve openff.units conflict by reinstalling openff packages...')
!mamba uninstall -y openff-toolkit openff-units openff-packmol
!mamba install -c conda-forge openff-toolkit openff-packmol -y
print('[FIX] Reinstallation of openff packages completed. Please re-run all cells after a runtime restart.')

conda_site_packages = "/usr/local/lib/python3.12/site-packages"
if conda_site_packages not in sys.path:
    sys.path.append(conda_site_packages)
from openff.packmol import pack_box

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from openff.toolkit.topology import Molecule

import openmm as mm
import openmm.app as app
from openmm import unit
from openmmforcefields.generators import GAFFTemplateGenerator

from openff.units import unit as openff_unit
from google.colab import files



# =================================================================================
# USER PARAMETERS
# =================================================================================

GLOBAL_RANDOM_SEED = 42          # fixed seed for reproducible embedding + dynamics
DP_PLA = 20                      # degree of polymerization for PLA chains
DP_PCL = 20                      # degree of polymerization for PCL chains
N_EMBED_ATTEMPTS = 20            # number of conformer-embedding attempts per chain

np.random.seed(GLOBAL_RANDOM_SEED)

# =================================================================================
# DIAGNOSTICS / TOOLCHAIN CHECKS (unchanged from the original script)
# =================================================================================

platforms = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print(f"[DIAGNOSTICS] Available OpenMM Platforms: {platforms}")
if 'CUDA' in platforms:
    print(" -> GPU Acceleration Enabled (CUDA ready)!")
else:
    print(" -> Running on CPU. (Tip: Change runtime type to T4 GPU for faster execution)")

try:
    test_mol = Molecule.from_smiles("CC(=O)O")
    print(" -> OpenFF Toolkit is active and ready for automatic GAFF parameterization.")
except Exception as e:
    print(f" -> OpenFF Error: {e}")

# ==============================================================================
# STEP 1: BUILD REPEAT-UNIT POLYMER CHAINS (replaces monomer-only construction)
# ==============================================================================
print(f"\n[1/6] Building PLA (DP={DP_PLA}) and PCL (DP={DP_PCL}) polymer chains...")

# Repeat-unit SMILES fragments (see module docstring for the chemistry).
# Each fragment is written so that simple string repetition + a trailing "O"
# cap produces a valid, correctly-connected linear polyester chain.
PLA_REPEAT_UNIT = "O[C@@H](C)C(=O)"   # -[O-CH(CH3)-C(=O)]-, isotactic (PLLA)
PCL_REPEAT_UNIT = "OCCCCCC(=O)"       # -[O-(CH2)5-C(=O)]-


def build_linear_polyester_smiles(repeat_unit_smiles, degree_of_polymerization):
    """
    Assemble the SMILES string for a linear, unbranched polyester chain by
    repeating `repeat_unit_smiles` `degree_of_polymerization` times and
    capping the ends with a terminal hydroxyl (-OH, implicit on the first
    atom) and a terminal carboxylic acid (-COOH, via a trailing "O").

    This is valid ONLY for repeat units written head-to-tail as
    "O...C(=O)" (i.e. starting with the ester oxygen and ending with the
    carbonyl carbon), which is the convention used for PLA_REPEAT_UNIT and
    PCL_REPEAT_UNIT above. Do not reuse this function for branched or
    differently-oriented repeat units without checking the resulting SMILES.

    Parameters
    ----------
    repeat_unit_smiles : str
        SMILES fragment for one repeat unit, head-to-tail oriented.
    degree_of_polymerization : int
        Number of repeat units in the chain (DP >= 1).

    Returns
    -------
    str
        SMILES string for the full linear chain.
    """
    if degree_of_polymerization < 1:
        raise ValueError("degree_of_polymerization must be >= 1")
    return (repeat_unit_smiles * degree_of_polymerization) + "O"


def smiles_to_rdkit_polymer(smiles, n_embed_attempts=N_EMBED_ATTEMPTS,
                             random_seed=GLOBAL_RANDOM_SEED, label=""):
    """
    Converts a SMILES string to an RDKit 3D molecule with explicit
    hydrogens and defined stereochemistry, with retry logic for conformer
    embedding (longer polymer chains are more prone to embedding failure
    with a single attempt than small monomers were).

    Parameters
    ----------
    smiles : str
        SMILES string of the (poly)molecule to embed.
    n_embed_attempts : int
        Number of randomized embedding attempts (ETKDGv3) before giving up.
    random_seed : int
        Seed for reproducible conformer generation.
    label : str
        Human-readable name used only in diagnostic print statements.

    Returns
    -------
    rdkit.Chem.Mol
        3D-embedded, UFF-optimized molecule with explicit hydrogens.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"[{label}] RDKit failed to parse SMILES:\n{smiles}")
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    params.useRandomCoords = True
    params.maxIterations = 1000

    conf_id = -1
    for attempt in range(n_embed_attempts):
        params.randomSeed = random_seed + attempt
        conf_id = AllChem.EmbedMolecule(mol, params)
        if conf_id == 0:
            break
    if conf_id != 0:
        raise RuntimeError(
            f"[{label}] Conformer embedding failed after {n_embed_attempts} "
            f"attempts. For long chains, consider building the polymer "
            f"incrementally (embed a short seed chain, then extend and "
            f"re-optimize in stages) rather than embedding the full chain "
            f"from a random start."
        )

    ff_converged = AllChem.UFFOptimizeMolecule(mol, maxIters=2000)
    if ff_converged != 0:
        print(f" -> [WARNING] [{label}] UFF optimization did not fully "
              f"converge (status={ff_converged}); geometry may be strained.")

    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
    return mol


def report_polymer_stats(mol, label, degree_of_polymerization):
    """
    Prints a sanity-check summary (formula, molecular weight, atom count,
    expected repeat count) so the user can visually confirm the built
    molecule is a chain of the intended length, not a monomer or a
    malformed structure, before it is packed into a simulation box.
    """
    formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
    mw = Descriptors.MolWt(mol)
    n_atoms_heavy = mol.GetNumAtoms() - sum(
        1 for atom in mol.GetAtoms() if atom.GetSymbol() == "H"
    )
    print(f" -> [{label}] DP={degree_of_polymerization} | Formula={formula} "
          f"| MW={mw:.1f} g/mol | Heavy atoms={n_atoms_heavy}")


print(" -> Constructing PLA chain SMILES...")
smiles_pla_chain = build_linear_polyester_smiles(PLA_REPEAT_UNIT, DP_PLA)
mol_pla = smiles_to_rdkit_polymer(smiles_pla_chain, label="PLA")
report_polymer_stats(mol_pla, "PLA", DP_PLA)

print(" -> Constructing PCL chain SMILES...")
smiles_pcl_chain = build_linear_polyester_smiles(PCL_REPEAT_UNIT, DP_PCL)
mol_pcl = smiles_to_rdkit_polymer(smiles_pcl_chain, label="PCL")
report_polymer_stats(mol_pcl, "PCL", DP_PCL)

# Export PDB for structural input / template generation, as in the original script
Chem.MolToPDBFile(mol_pla, "pla_chain.pdb")
files.download('pla_chain.pdb')
Chem.MolToPDBFile(mol_pcl, "pcl_chain.pdb")
files.download('pcl_chain.pdb')

# Convert RDKit objects -> OpenFF Molecule objects (with fallback flag)
openff_pla = Molecule.from_rdkit(mol_pla, allow_undefined_stereo=True)
openff_pcl = Molecule.from_rdkit(mol_pcl, allow_undefined_stereo=True)

print("[SUCCESS] OpenFF polymer-chain Molecules generated with assigned stereochemistry.")

# --- PACKMOL PACKING (unchanged in structure from the original script) ---
print("\n[1.5/6] Packing Multiple PLA and PCL Chains...")

selected_type = globals().pop('_GLOBAL_OVERRIDE_POLYMER_TYPE_', None)

if selected_type is None:
    n_pla_chains_current = getattr(globals().get('s_n_pla_chains'), 'value', 50)
    n_pcl_chains_current = getattr(globals().get('s_n_pcl_chains'), 'value', 50)
    selected_type = getattr(globals().get('s_polymer_type'), 'value', 'Blend')
else:
    n_pla_chains_current = getattr(globals().get('s_n_pla_chains'), 'value', 50)
    n_pcl_chains_current = getattr(globals().get('s_n_pcl_chains'), 'value', 50)

molecules_to_pack_unique = []
number_of_copies = []

if selected_type == 'Pure PLA':
    n_pcl_chains_current = 0
    molecules_to_pack_unique.append(openff_pla)
    number_of_copies.append(n_pla_chains_current)
    print(f"Selected: Pure PLA ({n_pla_chains_current} chains, DP={DP_PLA})")
elif selected_type == 'Pure PCL':
    n_pla_chains_current = 0
    molecules_to_pack_unique.append(openff_pcl)
    number_of_copies.append(n_pcl_chains_current)
    print(f"Selected: Pure PCL ({n_pcl_chains_current} chains, DP={DP_PCL})")
else:  # 'Blend'
    molecules_to_pack_unique.extend([openff_pla, openff_pcl])
    number_of_copies.extend([n_pla_chains_current, n_pcl_chains_current])
    print(f"Selected: Blend (PLA: {n_pla_chains_current} chains @ DP={DP_PLA}, "
          f"PCL: {n_pcl_chains_current} chains @ DP={DP_PCL})")

# Polymer chains are much larger than the earlier monomers, so a larger box
# is generally needed to pack the same number of chains without excessive
# initial overlap/strain. Adjust packing_box_size upward if pack_box fails
# to converge or minimization energies are pathologically high afterwards.
# Doubling the box size from 6.0 nm to 12.0 nm
packing_box_size = 12.0 * unit.nanometers

packed_system_pdb_file = "packed_system.pdb"
box_size_nm = packing_box_size.value_in_unit(unit.nanometer)
box_vectors_array = np.array([
    [box_size_nm, 0.0, 0.0],
    [0.0, box_size_nm, 0.0],
    [0.0, 0.0, box_size_nm]
])
box_vectors_packmol = openff_unit.Quantity(box_vectors_array, openff_unit.nanometer)

# Create a persistent working directory for Packmol
packmol_working_directory = '/content/packmol_temp'
os.makedirs(packmol_working_directory, exist_ok=True)

if not molecules_to_pack_unique:
    raise ValueError("No molecules selected for packing. Please select 'Pure PLA', 'Pure PCL', or 'Blend' using the dropdown.")

packed_system = pack_box(
    molecules=molecules_to_pack_unique,
    number_of_copies=number_of_copies,
    box_vectors=box_vectors_packmol,
    tolerance=2.0 * openff_unit.angstrom,
    working_directory=packmol_working_directory # Pass the persistent working directory
)

packed_system.to_file(packed_system_pdb_file, file_format="pdb")
files.download('packed_system.pdb')

print(f"[SUCCESS] Packed {n_pla_chains_current} PLA chains (DP={DP_PLA}) and "
      f"{n_pcl_chains_current} PCL chains (DP={DP_PCL}) into {packed_system_pdb_file}")

# ==============================================================================
# STEP 2: BUILD PERIODIC BOX & APPLY GAFF FORCE FIELD
# ==============================================================================
print("\n[2/6] Constructing Simulation Box and Assigning GAFF Parameters...")

pdb_packed = app.PDBFile(packed_system_pdb_file)

forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')

gaff_molecules_to_register = []
if openff_pla in molecules_to_pack_unique:
    gaff_molecules_to_register.append(openff_pla)
if openff_pcl in molecules_to_pack_unique:
    gaff_molecules_to_register.append(openff_pcl)

if not gaff_molecules_to_register:
    raise ValueError("No molecules selected for GAFF parameterization. Please select a polymer type.")

gaff = GAFFTemplateGenerator(molecules=gaff_molecules_to_register, forcefield='gaff-2.11')
forcefield.registerTemplateGenerator(gaff.generator)

system = forcefield.createSystem(
    pdb_packed.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=app.HBonds
)

# ==============================================================================
# STEP 3: INTEGRATOR & BAROSTAT SETUP
# ==============================================================================
print("\n[3/6] Setting up Langevin Integrator and Monte Carlo Barostat...")

initial_temp = 450.0 * unit.kelvin
pressure = 1.0 * unit.atmospheres
timestep = 2.0 * unit.femtoseconds

integrator = mm.LangevinMiddleIntegrator(
    initial_temp,
    1.0 / unit.picosecond,
    timestep
)
integrator.setRandomNumberSeed(GLOBAL_RANDOM_SEED)

barostat = mm.MonteCarloBarostat(pressure, initial_temp, 25)
barostat.setRandomNumberSeed(GLOBAL_RANDOM_SEED)
system.addForce(barostat)

try:
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'Precision': 'mixed'}
    print(" -> Utilizing NVIDIA GPU Acceleration (CUDA).")
except Exception:
    platform = mm.Platform.getPlatformByName('CPU')
    properties = {}
    print(" -> CUDA unavailable. Falling back to CPU Execution.")

simulation = app.Simulation(
    pdb_packed.topology,
    system,
    integrator,
    platform,
    properties
)
simulation.context.setPositions(pdb_packed.positions)

# ==============================================================================
# STEP 4: ENERGY MINIMIZATION & NVT MELT RELAXATION
# ==============================================================================
print("\n[4/6] Executing L-BFGS Energy Minimization...")
simulation.minimizeEnergy(maxIterations=1000)

print(" -> Running High-T NVT Melt Relaxation at 450 K...")
simulation.context.setVelocitiesToTemperature(initial_temp, GLOBAL_RANDOM_SEED)
simulation.step(50000)  # 100 ps

# ==============================================================================
# STEP 5: NPT STEP-WISE THERMAL ANNEALING (450 K -> 250 K)
# ==============================================================================
print("\n[5/6] Starting NPT Thermal Annealing Loop for Tg Extraction...")
print(" -> NOTE: see 'KNOWN LIMITATIONS' item 4 at the top of this file if "
      "running a Pure PCL case -- the range below may not reach PCL's real Tg.")

temps_to_simulate = np.arange(450, 230, -20)  # 450K, 430K, ..., 250K
annealing_results = []

for T in temps_to_simulate:
    target_temp = T * unit.kelvin
    integrator.setTemperature(target_temp)
    barostat.setDefaultTemperature(target_temp)

    simulation.step(20000)

    state = simulation.context.getState(getPositions=True, getEnergy=True)
    box_vectors = state.getPeriodicBoxVectors(asNumpy=True)
    volume = box_vectors[0][0] * box_vectors[1][1] * box_vectors[2][2]

    total_mass = sum([atom.element.mass.value_in_unit(unit.dalton)
                      for atom in pdb_packed.topology.atoms()])

    vol_cm3 = volume.value_in_unit(unit.centimeter**3)
    mass_g = total_mass * 1.66053906660e-24
    density_val = (mass_g / vol_cm3)

    annealing_results.append((T, density_val))
    print(f" -> T = {T:3d} K | Density = {density_val:.4f} g/cm3")

# ==============================================================================
# STEP 6: PLOT DENSITY vs TEMPERATURE AND EXTRACT Tg
# ==============================================================================
temps, densities = zip(*annealing_results)

n_pla = n_pla_chains_current
n_pcl = n_pcl_chains_current

base_title = "Mass Density vs. Temperature (NPT Cooling)"
if selected_type == 'Pure PLA':
    dynamic_part = f"Pure PLA ({n_pla} chains, DP={DP_PLA})"
elif selected_type == 'Pure PCL':
    dynamic_part = f"Pure PCL ({n_pcl} chains, DP={DP_PCL})"
else:
    dynamic_part = (f"PLA/PCL Blend (PLA: {n_pla} @ DP={DP_PLA}, "
                     f"PCL: {n_pcl} @ DP={DP_PCL})")

plot_title = f"{dynamic_part}\n{base_title}"

plt.figure(figsize=(7, 4.5), dpi=600)
plt.plot(temps, densities, 'o-', color='#1f77b4', linewidth=2, markersize=6)
plt.title(plot_title, fontsize=11)
plt.xlabel("Temperature (K)", fontsize=10)
plt.ylabel("Density (g/cm3)", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.gca().invert_xaxis()
plt.tight_layout()
plt.savefig("density_vs_temperature.png")
files.download('density_vs_temperature.png')
plt.show()

print("\n[SUCCESS] Pipeline completed successfully! Density plot generated.")

print("\n[6/6] Calculating Glass Transition Temperature (Tg)...")

temps_np = np.array(temps)
densities_np = np.array(densities)


def linear_function(x, m, c):
    return m * x + c


rubbery_indices = np.where((temps_np >= 350) & (temps_np <= 390))
temps_rubbery = temps_np[rubbery_indices]
densities_rubbery = densities_np[rubbery_indices]
popt_rubbery, pcov_rubbery = curve_fit(linear_function, temps_rubbery, densities_rubbery)
m_rubbery, c_rubbery = popt_rubbery
print(f"Rubbery Region Fit: Density = {m_rubbery:.5f} * Temp + {c_rubbery:.5f}")

glassy_indices = np.where((temps_np <= 310) & (temps_np >= 250))
temps_glassy = temps_np[glassy_indices]
densities_glassy = densities_np[glassy_indices]
popt_glassy, pcov_glassy = curve_fit(linear_function, temps_glassy, densities_glassy)
m_glassy, c_glassy = popt_glassy
print(f"Glassy Region Fit: Density = {m_glassy:.5f} * Temp + {c_glassy:.5f}")

if (m_rubbery - m_glassy) == 0:
    tg = np.nan
    print("Slopes are too similar, cannot determine intersection point.")
elif (m_rubbery - m_glassy) < 0:
    tg = (c_glassy - c_rubbery) / (m_rubbery - m_glassy)
else:
    print("Warning: Slopes do not cross in the expected manner for a glass "
          "transition (rubbery slope > glassy slope). Intersection might be "
          "outside data range -- treat the resulting Tg with caution.")
    tg = (c_glassy - c_rubbery) / (m_rubbery - m_glassy)

print(f"\nEstimated Glass Transition Temperature (Tg): {tg:.2f} K")

base_title_tg = "Density vs. Temperature with Linear Fits and Tg"
if selected_type == 'Pure PLA':
    dynamic_part_tg = f"Pure PLA ({n_pla} chains, DP={DP_PLA})"
elif selected_type == 'Pure PCL':
    dynamic_part_tg = f"Pure PCL ({n_pcl} chains, DP={DP_PCL})"
else:
    dynamic_part_tg = (f"PLA/PCL Blend (PLA: {n_pla} @ DP={DP_PLA}, "
                        f"PCL: {n_pcl} @ DP={DP_PCL})")

plot_title_tg = f"{dynamic_part_tg}\n{base_title_tg}"

x_fit = np.linspace(min(temps_np), max(temps_np), 100)
y_fit_rubbery = linear_function(x_fit, m_rubbery, c_rubbery)
y_fit_glassy = linear_function(x_fit, m_glassy, c_glassy)

fig = plt.figure(figsize=(8, 6), dpi=600)
plt.plot(temps_np, densities_np, 'o', color='#1f77b4', markersize=8, label='Simulation Data')
plt.plot(x_fit, y_fit_rubbery, '--', color='red', label=f'Rubbery Fit (T >= {min(temps_rubbery)}K)')
plt.plot(x_fit, y_fit_glassy, '--', color='green', label=f'Glassy Fit (T <= {max(temps_glassy)}K)')

if not np.isnan(tg):
    plt.axvline(x=tg, color='purple', linestyle=':', linewidth=2, label=f'Estimated Tg: {tg:.2f} K')
    plt.plot(tg, linear_function(tg, m_rubbery, c_rubbery), 'x', color='purple', markersize=10, markeredgewidth=2)

plt.title(plot_title_tg, fontsize=14)
plt.xlabel("Temperature (K)", fontsize=12)
plt.ylabel("Density (g/cm3)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.gca().invert_xaxis()
plt.tight_layout()
plt.savefig("density_vs_temperature_with_tg.png")
files.download('density_vs_temperature_with_tg.png')
plt.show()

print("\n[SUCCESS] Tg calculation and plot generated.")

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:15
🔁 Restarting kernel...
[+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   9%
conda-forge/noarch    14%[+] 0.3s
conda-forge/linux-64  16%
conda-forge/noarch    30%[+] 0.4s
conda-forge/linux-64  21%
conda-forge/noarch    40%[+] 0.5s
conda-forge/linux-64  25%
conda-forge/noarch    48%[+] 0.6s
conda-forge/linux-64  30%
conda-forge/noarch    58%[+] 0.7s
conda-forge/linux-64  35%
conda-forge/noarch    70%[+] 0.8s
conda-forge/linux-64  41%
conda-forge/noarch    76%[+] 0.9s
conda-forge/linux-64  44%
conda-forge/noarch    86%[+] 1.0s
conda-forge/linux-64  47%
conda-forge/noarch    94%conda-forge/noarch                                
[+] 1.1s
conda-forge/linux-64  53%[+] 1.2s
conda-forge/linux-64  61%[+] 1.3s
conda-forge/linux-64  66%[+]

At least one basic toolkit is required to handle SMARTS matching and file I/O. 
Please install at least one of the following basic toolkits:
The RDKit : A conda-installable version of the free and open source RDKit cheminformatics toolkit can be found at: https://anaconda.org/conda-forge/rdkit
AmberTools : The AmberTools toolkit (free and open source) can be found at https://anaconda.org/conda-forge/ambertools
Built-in Toolkit : This toolkit is installed with the Open Force Field Toolkit and does not require additional dependencies.
OpenEye Toolkit : The OpenEye Toolkits can be installed via `mamba install openeye-toolkits -c openeye`
OpenFF NAGL : See https://docs.openforcefield.org/projects/nagl/en/latest/installation.html



ImportError: /lib/x86_64-linux-gnu/libstdc++.so.6: version `CXXABI_1.3.15' not found (required by /usr/local/lib/python3.12/site-packages/scipy/optimize/_highspy/_core.cpython-312-x86_64-linux-gnu.so)